# Notebook 07: Transfer matrix (E11, hypothesis H1)

**Protocol v3.** Does the cross-country drop hold in every direction, and does shift-aimed augmentation reduce it?

| Task | Sources | Folds | Test set of the held-out source |
|---|---|---|---|
| Maize, 2 classes (streak virus, healthy) | TZ = Tanzania (D4), GH-C = Ghana CCMT (`d3m_adapt`), GH-K = Ghana KaraAgro (`d5_adapt`) | hold out each source; train on the other two | TZ: Phase 1 test partition; GH-C: `d3m_test`; GH-K: **`d5_test`, sealed, scored in Notebook 11** |
| Cassava, 3 classes | Ghana CCMT (`d3c_adapt`) → Uganda (D1 Phase 1 test partition) | one fold (the reverse of Phase 1) | D1 test partition |

Two arms per fold (Phase 1 augmentation; shift-aimed augmentation), EfficientNet-B0, seeds 17, 42, 2026: 24 runs. In each fold, 10% of each training source's images (by group) form that fold's selection set, used for early stopping and for the in-source score; the drop is in-source macro-F1 minus held-out-source macro-F1.

Only training, selection and the permitted test rows are loaded. `d5_test` is never loaded here. Runs save to `runs_v3/E11_*` and are resumable. **Runtime:** A100. Set `CROPS_TO_RUN` to split across two sessions.

In [ ]:
# 1. Setup, registration check, and the deviation record (see cell text below)
CROPS_TO_RUN = ['cassava', 'maize']
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, time, glob, tarfile, hashlib, datetime, platform, subprocess, gc, math, collections
import numpy as np, pandas as pd
import tensorflow as tf, keras
from sklearn.metrics import f1_score, accuracy_score
DRIVE = '/content/drive/MyDrive/ocdf'; WORK = '/content/working'
RUNS = f'{DRIVE}/runs_v3'; OUT = f'{DRIVE}/records/nb07'
for d in [RUNS, OUT, WORK]: os.makedirs(d, exist_ok=True)
reg_path = f'{DRIVE}/records/protocol_v3_registration.json'
reg_sha = open(reg_path + '.sha256').read().strip()
assert hashlib.sha256(open(reg_path, 'rb').read()).hexdigest() == reg_sha, 'registration changed'
reg = json.load(open(reg_path)); print('registration', reg['registered_utc'], reg_sha[:16])

# Deviation record: the registration JSON's text field holds a generator artefact; the structured fields were complete.
# The full registration text is attached here, before any Phase 2 experiment runs, and the defect is recorded.
REGISTRATION_TEXT = '# Protocol v3: pre-registered Phase 2, "Models that travel"\n\n**Project:** Offline On-Device System for Cassava and Maize Disease Detection Using CNNs\n**Builds on:** protocol v2 (manifest SHA-256 93463d32…0d97; frozen choices SHA-256 438e865d…7115). Phase 1 results are final and are reported unchanged as the baseline.\n**Status:** Written before any Phase 2 run. Notebook 06 writes this text to Google Drive with a timestamp and SHA-256 on its first run; every later notebook checks that hash. This version replaces an earlier draft that was never written to Drive.\n\n## 1. The question\n\nPhase 1 showed that a model scoring 0.85 accuracy on its own data scores 0.54 on the same crop in another country, and that source cues are perfectly separable in its features. Phase 2 asks four things, in order: how general is the drop (H1); what causes it (H2); what can be done about it on an offline device without expert labels (H3); and what the deployable model actually is, measured on a real phone (H4). Abstention under shift (H5) is kept from Phase 1 and extended.\n\n## 2. Honesty conditions\n\n- Phase 1\'s internal test and CCMT (D3) results have been seen. Results on those images in Phase 2 are **exploratory**. The protection against selection bias is this registration.\n- **KaraAgro (D5)** has never been opened. Half of it, split by group with seed 2026 (`d5_test`), is **sealed** and used once, at the end, as the **confirmatory** set for maize. The other half (`d5_adapt`) may be used for adaptation and as a training source in H1.\n- D3 is split the same way into `adapt` and `test` halves per crop. `d3_test` halves are never trained on or adapted to in any experiment. Cassava has no untouched source, so cassava results are exploratory throughout; the protocol records this.\n- Choices are made on selection or `adapt` data only. Nothing is tuned on a `test` half, the challenge set or D5.\n- Negative results are reported with the same prominence as positive ones.\n\n## 3. Hypotheses, experiments and endpoints\n\n### H1. The drop is general, not a one-off (transfer matrix; E11)\nTrain on one source, test on the others, every direction.\n- **Maize, two classes (streak virus vs healthy),** three field sources: Tanzania (D4), Ghana-CCMT (D3m), Ghana-KaraAgro (D5). Three folds, each holding out one source. Training uses the other two (D4 training partition; `adapt` halves of D3m and D5), selection on their selection subsets, testing on the held-out source\'s `test` half (for D4, the Phase 1 test partition).\n- **Cassava, three classes,** two sources: Uganda (D1) → Ghana (Phase 1 direction, already done) and Ghana (`d3c_adapt`) → Uganda (D1 test partition), the reverse direction.\n- Two arms per fold: **Phase 1 augmentation** and **shift-aimed augmentation** (random resized crop area 0.5–1.0, colour jitter ±25% and hue ±0.05, random erasing ≤25% with p=0.5, CutMix p=0.5). EfficientNet-B0, Phase 1 recipe, three seeds. 24 runs.\nEndpoints: macro-F1 matrix (in-source vs cross-source), the mean cross-source drop, and whether shift augmentation reduces it (paired by seed). The fold that holds out D5 is confirmatory for maize.\n\n### H2. The drop is caused by background and source cues (counterfactual; E12)\nUsing Ghana-CCMT images, whose plain studio backgrounds make leaf segmentation reliable (colour thresholding plus GrabCut, masks checked on sheets, images with failed masks excluded and counted):\n- **Background-only:** the leaf region replaced by its mean colour. If the Phase 1 model still predicts classes far from chance from the background alone, it reads backgrounds.\n- **Background swap:** the same segmented leaf pasted onto (a) a plain white background, (b) a plain dark background, (c) a Ugandan field background taken from D1 training images with the central region replaced, (d) a random non-plant texture. Prediction and confidence are compared across backgrounds for the same leaf.\nEndpoints: share of leaves whose predicted class changes when only the background changes; recall of CBB and CMD on the Ugandan-field background versus the plain original; background-only accuracy versus the majority-class rate. Exploratory (D3 seen), but causal in design.\n\n### H3. The gap can be closed on the device without expert labels (adaptation; E13)\nStarting from the frozen Phase 1 model (EfficientNet-B0, seed 42) and from the H1 shift-augmented model:\n- **AdaBN:** batch-normalisation statistics re-estimated on N unlabelled target images, nothing else changed.\n- **TENT:** entropy minimisation of the normalisation layers\' scale and shift on the same images.\n- **Gated self-training:** pseudo-labels from the frozen abstention rule (temperature and threshold from Phase 1), accepted predictions used to fine-tune the final blocks, two rounds.\n- **Labelled few-shot reference:** fine-tuning with N labelled local images, as the upper reference for what labels would buy.\nN ∈ {0, 10, 25, 50, 100, 200, 400} images per crop drawn group-wise from the `adapt` halves, three draws each. Evaluated on `d3c_test` (cassava) and `d3m_test` (maize) during development.\nDecision rule: the label-free method and N with the highest mean macro-F1 on `d3m_test` is **registered as the v3 adaptation** and applied once to maize D5: adapt on `d5_adapt`, score `d5_test`.\n**Primary confirmatory endpoint:** paired group-bootstrap difference (2,000 resamples) in macro-F1 (streak virus and healthy) on `d5_test` between the adapted and the unadapted Phase 1 model. "Confirmed" requires the 95% interval to exclude zero and the estimate to be ≥ +0.02. Secondary: the same on `d3c_test` and `d3m_test` (exploratory); a Ghana-vs-Uganda source probe on cassava features before and after adaptation.\n\n### H4. Full-integer INT8 fails at identifiable layers, and selective quantisation fixes it; on a real phone the model meets the budget (E14, E15)\n- LiteRT\'s quantisation debugger on the EfficientNet-B0 models reports per-layer error. Layers are ranked; the k worst are kept in float for k ∈ {0, 5, 10, 20, 40}, giving a size–accuracy curve on selection data. Decision rule: the smallest k whose INT8 file passes the Phase 1 budgets (≤10 MiB per crop, ≤20 MiB both; loss ≤2 pp accuracy and ≤0.02 macro-F1 versus FP32). MobileNetV2 (shift-augmented, from a B-arm of E11 with three classes/five classes) full-INT8 is the comparator.\n- **On-device measurement:** the official LiteRT `benchmark_model` binary run over USB on a physical Android phone, for FP16, full-INT8 and selective-INT8 files, one and four threads, 50 warm-up and 200 timed runs, three sessions, with the phone model, chipset, RAM, Android version and thermal state recorded. Plus the offline laptop benchmark. No application is built.\nEndpoints: layer-error ranking; the first k passing the budget; median and 90th-percentile latency and peak memory on the phone.\n\n### H5. Abstention under shift (E16)\n- Per-class thresholds (C1) chosen on the threshold-selection partition as registered in the earlier draft; energy score (C2) with the 95th-percentile rule.\n- Challenge set: near-unfamiliar cassava (CCMT brown spot, green mite), near-unfamiliar maize (CCMT fall armyworm, grasshopper, leaf beetle, leaf blight, leaf spot; KaraAgro fall armyworm), far-unfamiliar (CCMT cashew and tomato).\nEndpoints: false-acceptance rate and AUROC for max-softmax versus energy; AUROC of each score for predicting correctness on `test` halves and `d5_test`.\n\n## 4. Data added (Notebook 06)\n- D5 KaraAgro v4.1 (CC0, doi:10.7910/DVN/CXUMDS), registered subset `maize_streak_01.rar`, `healthy_01.rar`, `faw_01.zip`; MD5 checked.\n- CCMT unused folders (same release as D3), per-file SHA-256 checked.\n- Same inventory, duplicate and grouping rules as protocol v2. New images that copy any Phase 1 image (exact, or pHash distance ≤6) are excluded.\n- Partitions: `d3c_adapt`/`d3c_test`, `d3m_adapt`/`d3m_test`, `d5_adapt`/`d5_test` (group-wise, stratified by label, 50/50, seed 2026); `challenge_near_cassava`, `challenge_near_maize`, `challenge_far`. `d5_test` is sealed until `frozen_choices_v3.json` exists.\n\n## 5. Fixed decision rules\n1. v3 deployed model per crop: the H1 shift-augmented EfficientNet-B0 (seed 42) if its selection macro-F1 is within 0.02 of Phase 1, else the Phase 1 model.\n2. v3 deployed format: the smallest file passing all budgets on selection data among selective-INT8 (smallest k), MobileNetV2 full-INT8 (only if it is the deployed model), and FP16.\n3. v3 adaptation: as registered in H3; applied to D5 once.\n4. v3 abstention: C1 if "target met" for every class, else the Phase 1 global rule; C2 reported as a score only.\n5. Primary confirmatory endpoint: H3 on `d5_test`. Secondary confirmatory: H1 fold holding out D5.\n6. No experiment is added after registration. Deviations go in `protocol_v3_deviations.json` with date and reason.\n\n## 6. Notebooks\n| ID | Content | Notebook |\n|---|---|---|\n| E7 | Phase 2 data, splits, sealing | 06 (CPU) |\n| E11 | Transfer matrix, two arms | 07 (GPU) |\n| E12 | Background counterfactuals | 08 (GPU) |\n| E13 | Adaptation curves | 09 (GPU) |\n| E14 | Quantisation debugger, selective INT8, abstention rules, freeze v3 | 10 (GPU) |\n| E16 + final | Challenge set, test halves, D5 unsealed last | 11 (GPU) |\n| E15 | Phone and Mac benchmarks | local script |\n'
txt_path = f'{DRIVE}/records/protocol_v3_registration_text.md'
if not os.path.exists(txt_path):
    open(txt_path, 'w').write(REGISTRATION_TEXT)
    open(txt_path + '.sha256', 'w').write(hashlib.sha256(REGISTRATION_TEXT.encode()).hexdigest())
    dev_path = f'{DRIVE}/records/protocol_v3_deviations.json'
    devs = json.load(open(dev_path)) if os.path.exists(dev_path) else []
    devs.append({'date_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'notebook': '07',
                 'what': 'protocol_v3_registration.json (registered 2026-11-04T22:53 UTC) contains the structured registration fields '
                         '(hypotheses, primary endpoint, D5 archives, split rule) but its registration_text_markdown field holds the string '
                         "' + repr(REG_MD) + r' because of a bug in the notebook generator.",
                 'remedy': 'The full registration text is written to protocol_v3_registration_text.md with its SHA-256 at the start of Notebook 07, '
                           'before any Phase 2 training or evaluation. No experiment ran between the two timestamps except data preparation (Notebook 06).',
                 'affects_results': False})
    json.dump(devs, open(dev_path, 'w'), indent=1)
    print('registration text attached; deviation recorded')
assert hashlib.sha256(open(txt_path, 'rb').read()).hexdigest() == open(txt_path + '.sha256').read().strip()
gpus = tf.config.list_physical_devices('GPU'); assert gpus, 'Choose an A100 runtime'
for g in gpus:
    try: tf.config.experimental.set_memory_growth(g, True)
    except Exception: pass
gpu_name = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
env = {'notebook': '07_transfer_matrix', 'run_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'python': sys.version.split()[0],
       'tensorflow': tf.__version__, 'keras': keras.__version__, 'numpy': np.__version__, 'gpu': gpu_name, 'cpu_count': os.cpu_count(), 'platform': platform.platform()}
json.dump(env, open(f'{OUT}/environment_nb07.json', 'w'), indent=1); print(json.dumps(env, indent=1))

In [ ]:
# 2. Manifest v3 (integrity), working images, folds
seal = json.load(open(f'{DRIVE}/records/phase2_data_v3.json'))
mpath = f'{DRIVE}/manifests/{seal["manifest_v3"]}'
assert hashlib.sha256(open(mpath, 'rb').read()).hexdigest() == seal['sha256'], 'manifest_v3 changed since it was frozen'
M = pd.read_csv(mpath, dtype=str)
assert 'd5_test' in set(M.partition)
M = M[M.partition != 'd5_test'].copy()                      # SEALED: never loaded in this notebook
protocol2 = json.load(open(f'{DRIVE}/records/protocol_v2.json'))
for src in ['D1', 'D3', 'D4', 'D5', 'D6']:
    if not os.path.isdir(f'{WORK}/{src}'):
        with tarfile.open(f'{DRIVE}/working/working_{src}.tar') as tar: tar.extractall(WORK, filter='data')
SEEDS = [17, 42, 2026]; BATCH = 32; IMG = 224
RECIPE = {'head_lr': 1e-3, 'head_max_epochs': 5, 'finetune_lr': 1e-5, 'finetune_max_epochs': 25, 'patience': 5,
          'unfreeze_from': {'MobileNetV2': 'block_13', 'MobileNetV3Small': 'expanded_conv_8', 'EfficientNetB0': 'block6a'}}  # Phase 1 recipe (protocol v2)
ARMS = {'phase1': 'phase1', 'shift': 'shift'}
MAIZE2 = ['M01', 'M05']; CASSAVA3 = ['C01', 'C02', 'C03']
# source pools (rows that may be trained on)
POOL = {
  'TZ':   M[(M.source_id == 'D4') & (M.partition == 'train') & (M.research_label.isin(MAIZE2))],
  'GH-C': M[(M.partition == 'd3m_adapt')],
  'GH-K': M[(M.partition == 'd5_adapt')],
  'GH-C-cassava': M[(M.partition == 'd3c_adapt')],
}
TEST = {
  'TZ':   M[(M.source_id == 'D4') & (M.partition == 'test') & (M.research_label.isin(MAIZE2))],
  'GH-C': M[(M.partition == 'd3m_test')],
  'GH-K': None,                                              # d5_test: scored in Notebook 11 after the freeze
  'UG-cassava': M[(M.source_id == 'D1') & (M.partition == 'test')],
}
FOLDS = [('maize', 'TZ',   ['GH-C', 'GH-K'], MAIZE2),
         ('maize', 'GH-C', ['TZ', 'GH-K'],   MAIZE2),
         ('maize', 'GH-K', ['TZ', 'GH-C'],   MAIZE2),
         ('cassava', 'UG-cassava', ['GH-C-cassava'], CASSAVA3)]
def split_selection(df, frac=0.10, seed=2026):
    # 10% of groups (stratified by source and label) become the fold's selection set
    import random
    rng = random.Random(seed); sel = set()
    for (s, l), g in df.groupby(['source_id', 'research_label']):
        gids = sorted(set(g.group_id)); rng.shuffle(gids); sel |= set(gids[:max(1, int(len(gids) * frac))])
    return df[~df.group_id.isin(sel)], df[df.group_id.isin(sel)]
for crop, hold, trs, classes in FOLDS:
    tr = pd.concat([POOL[s] for s in trs]); print(f'fold hold-out {hold}: train pool {len(tr)} from {trs}; test {len(TEST[hold]) if TEST[hold] is not None else "sealed"}')

In [ ]:
# 3. Data pipeline with both augmentation sets (same code as registered) (all augmentation applies to training batches only)
def load_img(path, img):
    x = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    x = tf.image.resize_with_pad(x, img, img, method='bilinear')
    return tf.cast(tf.clip_by_value(tf.round(x), 0, 255), tf.uint8)
def base_ds(df, classes, img):
    paths = [f'{WORK}/{p}' for p in df.working_path]; y = [classes.index(l) for l in df.research_label]
    return tf.data.Dataset.from_tensor_slices((paths, y)).map(lambda p, l: (load_img(p, img), l), num_parallel_calls=tf.data.AUTOTUNE).cache()

def phase1_aug(seed):
    return keras.Sequential([keras.layers.RandomFlip('horizontal_and_vertical', seed=seed), keras.layers.RandomRotation(0.05, fill_mode='reflect', seed=seed),
                             keras.layers.RandomBrightness(0.1, value_range=(0, 255), seed=seed), keras.layers.RandomContrast(0.1, seed=seed)])

def random_resized_crop(x, img):
    B = tf.shape(x)[0]
    area = tf.random.uniform([B], 0.5, 1.0); ratio = tf.exp(tf.random.uniform([B], tf.math.log(0.75), tf.math.log(4 / 3)))
    w = tf.minimum(tf.sqrt(area * ratio), 1.0); h = tf.minimum(tf.sqrt(area / ratio), 1.0)
    x1 = tf.random.uniform([B]) * (1 - w); y1 = tf.random.uniform([B]) * (1 - h)
    boxes = tf.stack([y1, x1, y1 + h, x1 + w], axis=1)
    return tf.image.crop_and_resize(x, boxes, tf.range(B), [img, img])
def colour_jitter(x):
    B = tf.shape(x)[0]; sh = [B, 1, 1, 1]
    x = x + tf.random.uniform(sh, -0.25, 0.25) * 255.0                                           # brightness
    m = tf.reduce_mean(x, axis=[1, 2, 3], keepdims=True); x = m + (x - m) * tf.random.uniform(sh, 0.75, 1.25)   # contrast
    g = tf.image.rgb_to_grayscale(x); x = g + (x - g) * tf.random.uniform(sh, 0.75, 1.25)                     # saturation
    x = tf.clip_by_value(x, 0, 255)
    x = tf.map_fn(lambda im: tf.image.random_hue(im / 255.0, 0.05) * 255.0, x)                               # hue, per image
    return tf.clip_by_value(x, 0, 255)
def random_erasing(x, img, p=0.5, max_area=0.25):
    B = tf.shape(x)[0]
    area = tf.random.uniform([B], 0.02, max_area) * img * img; ratio = tf.exp(tf.random.uniform([B], tf.math.log(0.3), tf.math.log(3.3)))
    h = tf.minimum(tf.cast(tf.sqrt(area * ratio), tf.int32), img); w = tf.minimum(tf.cast(tf.sqrt(area / ratio), tf.int32), img)
    y1 = tf.cast(tf.random.uniform([B]) * tf.cast(img - h, tf.float32), tf.int32); x1 = tf.cast(tf.random.uniform([B]) * tf.cast(img - w, tf.float32), tf.int32)
    ys = tf.range(img)[None, :, None]; xs = tf.range(img)[None, None, :]
    inside = (ys >= y1[:, None, None]) & (ys < (y1 + h)[:, None, None]) & (xs >= x1[:, None, None]) & (xs < (x1 + w)[:, None, None])
    apply = tf.random.uniform([B]) < p
    mask = inside & apply[:, None, None]
    return tf.where(mask[..., None], 127.5, x)
def cutmix(x, y, sw, img, p=0.5):
    B = tf.shape(x)[0]; lam = tf.random.uniform([])
    cut = tf.sqrt(1.0 - lam); ch = tf.cast(cut * img, tf.int32); cw = ch
    cy = tf.random.uniform([], 0, img, dtype=tf.int32); cx = tf.random.uniform([], 0, img, dtype=tf.int32)
    y1 = tf.clip_by_value(cy - ch // 2, 0, img); y2 = tf.clip_by_value(cy + ch // 2, 0, img); x1 = tf.clip_by_value(cx - cw // 2, 0, img); x2 = tf.clip_by_value(cx + cw // 2, 0, img)
    ys = tf.range(img)[:, None]; xs = tf.range(img)[None, :]
    box = ((ys >= y1) & (ys < y2) & (xs >= x1) & (xs < x2))[None, :, :, None]
    xr, yr, sr = tf.reverse(x, [0]), tf.reverse(y, [0]), tf.reverse(sw, [0])
    lam_adj = 1.0 - tf.cast((y2 - y1) * (x2 - x1), tf.float32) / float(img * img)
    xm = tf.where(box, xr, x); ym = lam_adj * y + (1 - lam_adj) * yr; sm = lam_adj * sw + (1 - lam_adj) * sr
    do = tf.random.uniform([]) < p
    return tf.cond(do, lambda: (xm, ym, sm), lambda: (x, y, sw))

def train_ds(base, n_classes, seed, weights, img, aug_kind):
    w = tf.constant(weights, tf.float32); p1 = phase1_aug(seed)
    ds = base.shuffle(4096, seed=seed, reshuffle_each_iteration=True)
    ds = ds.map(lambda x, l: (tf.cast(x, tf.float32), tf.one_hot(l, n_classes), tf.gather(w, l)), num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH)
    if aug_kind == 'shift':
        def shift(x, y, sw):
            x = random_resized_crop(x, img); x = colour_jitter(x); x = random_erasing(x, img)
            x, y, sw = cutmix(x, y, sw, img)
            return tf.clip_by_value(p1(x, training=True), 0, 255), y, sw
        ds = ds.map(shift, num_parallel_calls=tf.data.AUTOTUNE)
    else:
        ds = ds.map(lambda x, y, sw: (tf.clip_by_value(p1(x, training=True), 0, 255), y, sw), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)
def eval_ds(base, n_classes):
    return base.map(lambda x, l: (tf.cast(x, tf.float32), tf.one_hot(l, n_classes))).batch(BATCH).prefetch(tf.data.AUTOTUNE)

In [ ]:
# 4. Models (as Phase 1, with a variable input size) and the macro-F1 stopper
def build_model(arch, n_classes, img):
    inp = keras.Input((img, img, 3), name='image_0_255')
    if arch == 'MobileNetV2':
        x = keras.layers.Rescaling(1 / 127.5, offset=-1, name='mnv2_scale')(inp)
        base = keras.applications.MobileNetV2(include_top=False, weights='imagenet', input_shape=(img, img, 3))
    else:
        x = inp; base = keras.applications.EfficientNetB0(include_top=False, weights='imagenet', input_shape=(img, img, 3))
    base.trainable = False
    x = base(x, training=False); x = keras.layers.GlobalAveragePooling2D(name='gap')(x); x = keras.layers.Dropout(0.2, name='dropout')(x)
    return keras.Model(inp, keras.layers.Dense(n_classes, activation='softmax', name='probs')(x), name=f'{arch}_{n_classes}cls_{img}'), base
def unfreeze_final_blocks(base, arch):
    prefix = RECIPE['unfreeze_from'][arch]; names = [l.name for l in base.layers]
    idx = next((i for i, n in enumerate(names) if n.startswith(prefix)), int(len(names) * 0.75))
    base.trainable = True
    for l in base.layers[:idx]: l.trainable = False
    return {'first_trainable_layer': names[idx], 'trainable_backbone_layers': len(names) - idx, 'total_backbone_layers': len(names)}
class MacroF1Stopper(keras.callbacks.Callback):
    def __init__(self, sel_ds, y_sel, patience, log, phase):
        super().__init__(); self.sel_ds, self.y_sel, self.patience, self.log, self.phase = sel_ds, y_sel, patience, log, phase
        self.best, self.best_w, self.wait, self.t0 = -1.0, None, 0, None
    def on_epoch_begin(self, epoch, logs=None): self.t0 = time.time()
    def on_epoch_end(self, epoch, logs=None):
        p = self.model.predict(self.sel_ds, verbose=0).argmax(1); f1 = f1_score(self.y_sel, p, average='macro'); acc = accuracy_score(self.y_sel, p)
        self.log.append({'phase': self.phase, 'epoch': epoch + 1, 'train_loss': float(logs.get('loss', np.nan)), 'select_macro_f1': float(f1), 'select_accuracy': float(acc), 'seconds': round(time.time() - self.t0, 1)})
        print(f'  {self.phase} epoch {epoch+1}: loss {logs.get("loss", 0):.4f}  selection macro-F1 {f1:.4f}  acc {acc:.4f}  ({self.log[-1]["seconds"]}s)')
        if f1 > self.best + 1e-6: self.best, self.best_w, self.wait = f1, self.model.get_weights(), 0
        else:
            self.wait += 1
            if self.wait >= self.patience: self.model.stop_training = True
    def on_train_end(self, logs=None):
        if self.best_w is not None: self.model.set_weights(self.best_w)

In [ ]:
# 5. Pipeline check: one shift-augmented batch saved as a sheet
from PIL import Image
df = POOL['GH-C-cassava'].sample(64, random_state=1); keras.utils.set_random_seed(17)
xb, yb, sb = next(iter(train_ds(base_ds(df, CASSAVA3, IMG), 3, 17, [1, 1, 1], IMG, 'shift')))
tiles = [Image.fromarray(xb[i].numpy().astype(np.uint8)).resize((128, 128)) for i in range(16)]
sheet = Image.new('RGB', (128 * 8, 128 * 2)); [sheet.paste(t, ((k % 8) * 128, (k // 8) * 128)) for k, t in enumerate(tiles)]
sheet.save(f'{OUT}/augmentation_sheet_shift.png'); print('batch', xb.shape, 'label rows (CutMix mixes them):', np.round(yb.numpy().sum(1)[:4], 2))

In [ ]:
# 6. E11 runs (resumable). Class weights from each fold's training counts (Phase 1 formula, cap 5).
def class_weights(counts, cap=5.0):
    n = sum(counts.values()); k = len(counts); return {c: min(n / (k * counts[c]), cap) for c in counts}
def run_fold(crop, hold, trs, classes, arm, seed):
    rid = f'E11_{crop}_holdout-{hold}_{arm}_s{seed}'; rdir = f'{RUNS}/{rid}'
    if os.path.exists(f'{rdir}/done.json'): print('done already:', rid); return
    os.makedirs(rdir, exist_ok=True)
    pool = pd.concat([POOL[s] for s in trs]); tr, se = split_selection(pool)
    key = (crop, hold)
    if key not in CACHE: CACHE[key] = (base_ds(tr, classes, IMG), eval_ds(base_ds(se, classes, IMG), len(classes)), tr, se)
    _, sel_ds, tr, se = CACHE[key]
    cw = class_weights(tr.research_label.value_counts().to_dict()); weights = [cw[c] for c in classes]
    keras.utils.set_random_seed(seed); keras.backend.clear_session()
    tds = train_ds(CACHE[key][0], len(classes), seed, weights, IMG, arm)
    y_sel = np.array([classes.index(l) for l in se.research_label])
    model, base = build_model('EfficientNetB0', len(classes), IMG); log = []; t0 = time.time()
    print(f'=== {rid}: train {len(tr)}, selection {len(se)}, weights {dict(zip(classes, np.round(weights, 3)))} ===')
    model.compile(optimizer=keras.optimizers.Adam(RECIPE['head_lr']), loss='categorical_crossentropy', metrics=['accuracy'])
    model.fit(tds, epochs=RECIPE['head_max_epochs'], verbose=0, callbacks=[MacroF1Stopper(sel_ds, y_sel, RECIPE['patience'], log, 'head')])
    ufz = unfreeze_final_blocks(base, 'EfficientNetB0')
    model.compile(optimizer=keras.optimizers.Adam(RECIPE['finetune_lr']), loss='categorical_crossentropy', metrics=['accuracy'])
    st = MacroF1Stopper(sel_ds, y_sel, RECIPE['patience'], log, 'finetune'); st.best = max(r['select_macro_f1'] for r in log); st.best_w = model.get_weights()
    model.fit(tds, epochs=RECIPE['finetune_max_epochs'], verbose=0, callbacks=[st])
    model.save(f'{rdir}/best_model.keras'); pd.DataFrame(log).to_csv(f'{rdir}/history.csv', index=False)
    sel_p = model.predict(sel_ds, verbose=0); in_f1 = float(f1_score(y_sel, sel_p.argmax(1), average='macro'))
    info = {'run_id': rid, 'experiment': 'E11', 'crop': crop, 'holdout': hold, 'train_sources': trs, 'arm': arm, 'seed': seed, 'classes': classes,
            'n_train': len(tr), 'n_select': len(se), 'in_source_select_macro_f1': in_f1, 'in_source_select_accuracy': float(accuracy_score(y_sel, sel_p.argmax(1))),
            'class_weights': cw, 'unfreeze': ufz, 'epochs_run': {'head': sum(r['phase'] == 'head' for r in log), 'finetune': sum(r['phase'] == 'finetune' for r in log)},
            'minutes': round((time.time() - t0) / 60, 1), 'environment': env, 'registration_sha256': reg_sha}
    te = TEST[hold]
    if te is not None:
        y_te = np.array([classes.index(l) for l in te.research_label]); p = model.predict(eval_ds(base_ds(te, classes, IMG), len(classes)), verbose=0)
        np.save(f'{rdir}/test_logits_proxy_probs.npy', p)
        pd.DataFrame({'sample_id': te.sample_id, 'group_id': te.group_id, 'y': y_te, 'pred': p.argmax(1), 'conf': p.max(1)}).to_csv(f'{rdir}/test_predictions.csv', index=False)
        info.update({'test_set': hold, 'n_test': len(te), 'test_macro_f1': float(f1_score(y_te, p.argmax(1), average='macro')),
                     'test_accuracy': float(accuracy_score(y_te, p.argmax(1))),
                     'test_per_class_recall': {c: float((p.argmax(1)[y_te == i] == i).mean()) for i, c in enumerate(classes)}})
        info['drop_macro_f1'] = in_f1 - info['test_macro_f1']
        print(f'>>> {rid}: in-source {in_f1:.4f}  held-out {info["test_macro_f1"]:.4f}  drop {info["drop_macro_f1"]:+.4f}  ({info["minutes"]} min)')
    else:
        info.update({'test_set': 'd5_test (sealed; scored in Notebook 11)'}); print(f'>>> {rid}: in-source {in_f1:.4f}; held-out set sealed ({info["minutes"]} min)')
    json.dump(info, open(f'{rdir}/done.json', 'w'), indent=1, default=float)
    del model, base, tds; gc.collect()
CACHE = {}
for crop, hold, trs, classes in FOLDS:
    if crop not in CROPS_TO_RUN: continue
    for arm in ARMS:
        for seed in SEEDS: run_fold(crop, hold, trs, classes, arm, seed)
    CACHE.clear(); gc.collect()
print('finished requested runs')

In [ ]:
# 7. Transfer matrix summary and results file
import zipfile
rows = [json.load(open(p)) for p in sorted(glob.glob(f'{RUNS}/E11_*/done.json'))]
S = pd.DataFrame([{k: r.get(k) for k in ['run_id', 'crop', 'holdout', 'arm', 'seed', 'n_train', 'n_test', 'in_source_select_macro_f1', 'test_macro_f1', 'test_accuracy', 'drop_macro_f1', 'minutes']} for r in rows])
S.to_csv(f'{OUT}/e11_runs.csv', index=False)
agg = S.groupby(['crop', 'holdout', 'arm']).agg(runs=('seed', 'count'), in_source=('in_source_select_macro_f1', 'mean'), held_out=('test_macro_f1', 'mean'),
                                                held_out_sd=('test_macro_f1', 'std'), drop=('drop_macro_f1', 'mean')).round(4)
agg.to_csv(f'{OUT}/e11_transfer_matrix.csv'); print(agg)
# paired by seed: shift minus phase1 on the held-out source
P = S.pivot_table(index=['crop', 'holdout', 'seed'], columns='arm', values='test_macro_f1')
if {'phase1', 'shift'} <= set(P.columns):
    P['shift_minus_phase1'] = P['shift'] - P['phase1']; print(P.groupby(['crop', 'holdout']).shift_minus_phase1.agg(['mean', 'std', 'count']).round(4))
    P.to_csv(f'{OUT}/e11_arm_difference_by_seed.csv')
res = '/content/nb07_results.zip'
with zipfile.ZipFile(res, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in [f'{OUT}/environment_nb07.json', f'{OUT}/e11_runs.csv', f'{OUT}/e11_transfer_matrix.csv', f'{OUT}/e11_arm_difference_by_seed.csv', f'{OUT}/augmentation_sheet_shift.png',
              f'{DRIVE}/records/protocol_v3_deviations.json', f'{DRIVE}/records/protocol_v3_registration_text.md.sha256']:
        if os.path.exists(p): z.write(p, os.path.basename(p))
    for r in rows:
        for f in ['done.json', 'history.csv', 'test_predictions.csv']:
            if os.path.exists(f'{RUNS}/{r["run_id"]}/{f}'): z.write(f'{RUNS}/{r["run_id"]}/{f}', f'runs_v3/{r["run_id"]}/{f}')
from google.colab import files
files.download(res); print(len(rows), 'of 24 runs finished')

### When it finishes
The run records are written to Google Drive under `ocdf/records/` and also downloaded as `nb07_results.zip`. The copies used for the published results are in this repository under `records/nb07/`.